In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier

# 1. Load and Clean the Data
train_df = pd.read_csv('Training.csv') 


train_df = train_df.fillna(0)
train_df = train_df.drop_duplicates()
X_train = train_df.drop('prognosis', axis=1) 
y_train = train_df['prognosis']              

# 2. Train the "Scout" Model to analyze symptom importance
scout_model = RandomForestClassifier(n_estimators=100, random_state=42)
scout_model.fit(X_train, y_train)

# 3. Dynamically Filter Features (The Goldilocks Zone)
importances = scout_model.feature_importances_
symptom_importance = pd.DataFrame({
    'Symptom': X_train.columns,
    'Importance': importances
})

# Set a threshold: Keep symptoms with at least 0.5% importance (0.005)
THRESHOLD = 0.003

# Filter the dataset based on the threshold
important_symptoms_df = symptom_importance[symptom_importance['Importance'] >= THRESHOLD]
top_symptoms = important_symptoms_df['Symptom'].tolist()

print(f"Original symptoms: {len(X_train.columns)}")
print(f"Symptoms kept after dynamic filtering: {len(top_symptoms)}\n")

# 4. Create the Optimized Dataset
X_train_optimized = X_train[top_symptoms]

# 5. Train the FINAL model
# 5. Train the FINAL model
# Remove max_depth. Instead, use min_samples_leaf to prevent perfect memorization
# We remove min_samples_leaf and lower the estimators to make the trees more decisive
final_model = RandomForestClassifier(
    n_estimators=50, 
    max_depth=10, # Allowing deeper trees lets it find more specific patterns
    random_state=42
)
final_model.fit(X_train_optimized, y_train)
# 6. The Prediction Function
def predict_smart(user_symptoms):
    input_data = np.zeros(len(top_symptoms))
    
    recognized_symptoms = []
    for symptom in user_symptoms:
        if symptom in top_symptoms:
            index = top_symptoms.index(symptom)
            input_data[index] = 1
            recognized_symptoms.append(symptom)
        else:
            print(f"  [!] Symptom '{symptom}' was filtered out as noise or is mispelled.")
            
    input_df = pd.DataFrame([input_data], columns=top_symptoms)
    
    probabilities = final_model.predict_proba(input_df)[0]
    disease_probs = list(zip(final_model.classes_, probabilities))
    disease_probs.sort(key=lambda x: x[1], reverse=True)
    
    print(f"\nAnalyzing recognized symptoms: {recognized_symptoms}")
    print("Differential Diagnosis:")
    
    # Print the top 3 possibilities
    for disease, prob in disease_probs[:3]:
        if prob > 0:
            print(f"- {disease}: {prob * 100:.1f}%")
    print("-" * 40)

# --- Test it out! ---
my_symptoms = [ 'skin_rash','itching']
predict_smart(my_symptoms)

my_other_symptoms = ['cough','high_fever','loss_of_smell','loss_of_taste']
predict_smart(my_other_symptoms)

Original symptoms: 58
Symptoms kept after dynamic filtering: 58


Analyzing recognized symptoms: ['skin_rash', 'itching']
Differential Diagnosis:
- Psoriasis: 30.2%
- Eczema: 25.9%
- Fungal Infection: 19.1%
----------------------------------------

Analyzing recognized symptoms: ['cough', 'high_fever', 'loss_of_smell', 'loss_of_taste']
Differential Diagnosis:
- COVID-19: 44.4%
- Common Cold: 13.5%
- Bronchitis: 9.8%
----------------------------------------


In [2]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# ... [Keep all your previous code above this] ...

print("\n" + "="*50)
print("PHASE 3: MODEL EVALUATION ON TESTING DATA")
print("="*50)

# 1. Load the Testing dataset
# Replace 'Testing.csv' with your actual testing file name
test_df = pd.read_csv('Training.csv') 

# 2. Clean the Data (Same as Training)
test_df = test_df.fillna(0)

# 3. Separate Features and Target
# CRITICAL STEP: We only select the 'top_symptoms' the model was trained on!
X_test = test_df[top_symptoms]
y_test = test_df['prognosis']

# 4. Make Predictions on the unseen data
y_pred = final_model.predict(X_test)

# 5. Calculate Overall Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"\nOverall Model Accuracy: {accuracy * 100:.2f}%\n")

# 6. Generate a Classification Report
# This shows Precision, Recall, and F1-Score for EVERY disease
print("Detailed Classification Report:")
print(classification_report(y_test, y_pred))

# --- BTP REPORT BONUS: Confusion Matrix Visual ---
def plot_confusion_matrix(y_true, y_predicted, classes):
    plt.figure(figsize=(12, 10))
    cm = confusion_matrix(y_true, y_predicted)
    
    # Create a visually appealing heatmap
    sns.heatmap(cm, annot=False, cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.title('Disease Prediction Confusion Matrix')
    plt.ylabel('Actual Disease (Ground Truth)')
    plt.xlabel('Predicted Disease')
    plt.xticks(rotation=90)
    plt.tight_layout()
    
    # Save the plot as an image so you can put it in your project report!
    plt.savefig('confusion_matrix.png')
    print("\n[Success] Confusion Matrix saved as 'confusion_matrix.png' in your folder!")
    plt.show()

# Uncomment the line below to generate the graph!
# plot_confusion_matrix(y_test, y_pred, final_model.classes_)


PHASE 3: MODEL EVALUATION ON TESTING DATA

Overall Model Accuracy: 97.98%

Detailed Classification Report:
                           precision    recall  f1-score   support

                     Acne       0.92      0.96      0.94       318
                  Allergy       1.00      0.99      0.99       331
                   Asthma       0.99      0.99      0.99       327
               Bronchitis       0.95      0.97      0.96       327
                 COVID-19       0.99      0.93      0.96       301
              Common Cold       0.98      0.99      0.98       309
Conjunctivitis (Pink Eye)       0.99      1.00      0.99       313
                   Eczema       0.99      0.97      0.98       354
         Fungal Infection       0.99      0.98      0.99       333
                     GERD       1.00      0.98      0.99       315
          Gastroenteritis       0.99      1.00      0.99       318
             Hypertension       1.00      0.98      0.99       308
          Influenza 

In [3]:
import joblib

# Save the trained Random Forest and the symptom list
joblib.dump(final_model, 'triage_model.pkl')
joblib.dump(top_symptoms, 'symptoms_list.pkl')
print("Model saved for the API!")

Model saved for the API!
